In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Using existing Dataproc Session (configuration changes may not be applied): https://console.cloud.google.com/dataproc/interactive/asia-southeast1/sc-20260403-081805-s4sqls?project=project-7f16ff1d-9026-4799-bfa


✅ Spark Connect đã sẵn sàng cho dự án: project-7f16ff1d-9026-4799-bfa


In [ ]:

PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET = "amazon-reviews-lakehouse-data"
DATASET_ID = "gold_zone"  
MASTER_PATH = f"gs://{BUCKET}/silver-zone/silver-master-table"
CURRENT_TS = 1775260800  # 04/2026

from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from google.cloud import bigquery

# Khởi tạo Spark
spark = DataprocSparkSession.builder.getOrCreate()

Using existing Dataproc Session (configuration changes may not be applied): https://console.cloud.google.com/dataproc/interactive/asia-southeast1/sc-20260403-081805-s4sqls?project=project-7f16ff1d-9026-4799-bfa


In [ ]:
# Khởi tạo BigQuery Client để kiểm tra dataset
bq_client = bigquery.Client(project=PROJECT_ID)

def check_and_create_dataset():
    dataset_full_id = f"{PROJECT_ID}.{DATASET_ID}"
    try:
        bq_client.get_dataset(dataset_full_id)
        print(f"✅ Tuyệt vời! Dataset '{DATASET_ID}' đã sẵn sàng.")
    except Exception:
        print(f"⚠️ Không tìm thấy '{DATASET_ID}'. Đang tự động tạo mới...")
        dataset = bigquery.Dataset(dataset_full_id)
        dataset.location = "asia-southeast1" 
        bq_client.create_dataset(dataset, timeout=30)
        print(f"✅ Đã tạo thành công Dataset: {DATASET_ID}")

check_and_create_dataset()

✅ Tuyệt vời! Dataset 'gold_zone' đã sẵn sàng.


In [12]:
def build_gold_user_dna_final():
    print(f"🚀 Khởi động tiến trình đẩy dữ liệu vào BigQuery: {DATASET_ID}")

    # 1. Đọc dữ liệu Master
    df_master = spark.read.parquet(MASTER_PATH)

    # 2. Tính toán DNA người dùng (Đã fix lỗi DataType Mismatch)
    print("📊 Đang tổng hợp DNA người dùng...")
    user_base = df_master.groupBy("user_id").agg(
        F.count("parent_asin").alias("total_reviews_count"),
        F.avg("rating").alias("avg_user_rating"),
        F.avg("price").alias("price_sensitivity_avg"),
        F.countDistinct("main_category").alias("diversity_score"),
        F.avg(F.col("verified_purchase").cast("float")).alias("verified_purchase_ratio"),
        F.max("timestamp").alias("last_active_ts")
    )

    # 3. Tính Recency (Ép kiểu sang Long để tránh lỗi trừ INT với TIMESTAMP)
    print("📈 Tính toán Recency & Phân loại User...")
    user_enriched = user_base.withColumn(
        "user_recency_days",
        (F.lit(CURRENT_TS).cast("long") - F.col("last_active_ts").cast("long")) / 86400
    )

    # Phân bậc (Top 10% là Power User)
    window_spec = Window.orderBy(F.col("total_reviews_count").desc())
    user_segmented = user_enriched.withColumn(
        "rank_pct", F.percent_rank().over(window_spec)
    ).withColumn(
        "user_activity_level",
        F.when(F.col("rank_pct") <= 0.1, "Power User")
         .when(F.col("rank_pct") <= 0.5, "Active User")
         .otherwise("New User")
    ).drop("rank_pct", "last_active_ts")

    # 4. Xác định Category Affinity (Ngành hàng yêu thích)
    print("🛍️ Xác định ngành hàng yêu thích nhất...")
    cat_counts = df_master.groupBy("user_id", "main_category").count()
    cat_window = Window.partitionBy("user_id").orderBy(F.col("count").desc())

    primary_cat = cat_counts.withColumn("rank", F.row_number().over(cat_window)) \
        .filter(F.col("rank") == 1) \
        .select("user_id", F.col("main_category").alias("category_affinity"))

    # 5. Join và lưu vào BigQuery
    final_user_dna = user_segmented.join(primary_cat, "user_id", "left")

    print(f"💾 Đang ghi dữ liệu vào: {PROJECT_ID}.{DATASET_ID}.user_dna_profiles")

    final_user_dna.write.format("bigquery") \
        .option("table", f"{PROJECT_ID}.{DATASET_ID}.user_dna_profiles") \
        .option("temporaryGcsBucket", BUCKET) \
        .mode("overwrite") \
        .save()

    print(f"✨ HOÀN TẤT! Hãy kiểm tra Dataset '{DATASET_ID}' trên BigQuery Console.")
    final_user_dna.show(5)

# Thực thi lần cuối
build_gold_user_dna_final()

🚀 Khởi động tiến trình đẩy dữ liệu vào BigQuery: gold_zone
📊 Đang tổng hợp DNA người dùng...
📈 Tính toán Recency & Phân loại User...
🛍️ Xác định ngành hàng yêu thích nhất...
💾 Đang ghi dữ liệu vào: project-7f16ff1d-9026-4799-bfa.gold_zone.user_dna_profiles


  0%|           0/268 Tasks

✨ HOÀN TẤT! Hãy kiểm tra Dataset 'gold_zone' trên BigQuery Console.


  0%|           0/268 Tasks

+--------------------+-------------------+-----------------+---------------------+---------------+-----------------------+------------------+-------------------+--------------------+
|             user_id|total_reviews_count|  avg_user_rating|price_sensitivity_avg|diversity_score|verified_purchase_ratio| user_recency_days|user_activity_level|   category_affinity|
+--------------------+-------------------+-----------------+---------------------+---------------+-----------------------+------------------+-------------------+--------------------+
|AH665SQ6SQF6DXAGY...|              13128|4.095749542961609|    45.00068547259065|             30|   0.028641072516758074|1114.3075694444444|         Power User|         Amazon Home|
|AGZZXSMMS4WRHHJRB...|              13228|4.857272452373753|   35.257187082007455|             31|   0.006652555185969156|1114.5922916666666|         Power User|      AMAZON FASHION|
|AEYVPPWR4CIKWX4BG...|               7718|4.437030318735424|   27.362999779798837|   